# Loading Data

In [127]:
# USE THIS WHEN RUNNING ON COLAB

"""
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/combined_ipl_mbm_bat.csv')"""


'\nimport kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf = pd.read_csv(path + \'/combined_ipl_mbm_bat.csv\')'

In [31]:
# USE THIS WHEN RUNNING LOCALLY

import pandas as pd
import numpy as np

df = pd.read_csv(r"../../data/mbm/all_league_bat_final.csv")

In [32]:
# Convert date to datetime format
df['date'] = pd.to_datetime(df['date'])

# Calculate boundaries from fours and sixes
df['boundaries'] = df['fours'] + df['sixes']

# Remove unnecessary columns
df.drop(columns=['fours', 'sixes', 'sr', 'bowler_id', 'bowler'], inplace=True)

# Load player details
players_detail = pd.read_csv(r"D:\ED\FIFS_IPLungit\data\register_data\people3.csv ")

# Create a clean mapping dictionary by filtering out NaN values
valid_players = players_detail.dropna(subset=['key_cricinfo'])
name_corrector = dict(zip(valid_players['key_cricinfo'], valid_players['unique_name']))

# Apply the mapping without any fallback - missing values will remain as NaN
df['batter'] = df['batter_id'].map(name_corrector)

# # Optionally report missing mappings but don't replace them
# missing_batters = df['batter'].isna()
# if missing_batters.any():
#     print(f"Found {missing_batters.sum()} batters without name mappings")

In [33]:
df.columns

Index(['match_id', 'date', 'inns', 'batter', 'batter_id', 'bat_team',
       'bowl_team', 'runs', 'balls', 'isOut', 'wicketType', 'venue', 'league',
       'boundaries'],
      dtype='object')

# Career Stats

In [34]:
df['isOut'] = df['isOut'].replace('DNB', 'False')
df['isOut'] = df['isOut'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

In [35]:
# Sort data by batter and match date
df.sort_values(by=['batter_id', 'date'], inplace=True)

# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby(['batter_id', 'league'])['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby(['batter_id', 'league'])['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby(['batter_id', 'league']).cumcount()
df['cumulative_balls'] = df.groupby(['batter_id', 'league'])['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby(['batter_id', 'league'])['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['career_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['career_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['career_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['career_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['career_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
career_cols = ['career_avg', 'career_bfpm', 'career_sr', 'career_rpm', 'career_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in career_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter','league']).head(1).index, career_cols] = pd.NA

In [36]:
df[(df['batter_id'] == 793463)].head()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,wicketType,venue,league,boundaries,career_avg,career_rpm,career_sr,career_bpm,career_bfpm
46478,1063062,2016-11-13,2,Rashid Khan,793463,Cumilla Warriors,Khulna Tigers,10,7,1,run out,Shere Bangla National Stadium,BPL,2,NaN,NaN,NaN,NaN,NaN
46213,1063050,2016-11-14,2,Rashid Khan,793463,Cumilla Warriors,Dhaka Capital,2,3,1,caught,Shere Bangla National Stadium,BPL,0,10.0,10.00,142.857143,2.000000,7.000000
46559,1063066,2016-11-18,1,Rashid Khan,793463,Cumilla Warriors,Rangpur Riders,0,0,0,DNB,Zahur Ahmed Chowdhury Stadium,BPL,0,6.0,6.00,120.000000,1.000000,5.000000
46766,1063075,2016-11-26,2,Rashid Khan,793463,Cumilla Warriors,Dhaka Capital,3,3,1,stumped,Shere Bangla National Stadium,BPL,0,6.0,4.00,120.000000,0.666667,3.333333
46859,1063079,2016-11-29,2,Rashid Khan,793463,Cumilla Warriors,Fortune Barishal,0,0,0,DNB,Shere Bangla National Stadium,BPL,0,5.0,3.75,115.384615,0.500000,3.250000


In [37]:
df.sample()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,wicketType,venue,league,boundaries,career_avg,career_rpm,career_sr,career_bpm,career_bfpm
10563,530341,2012-03-04,2,PJ Malan,277390,Titans,Cape Cobras,48,41,1,caught,Newlands,CSA,6,12.0,12.0,75.0,2.0,16.0


In [39]:
df[df['batter_id'] == 44936].info() # NOTE - Career columns can have one row of null values for each batter


<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 161 to 102592
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   match_id     315 non-null    int64         
 1   date         315 non-null    datetime64[ns]
 2   inns         315 non-null    int64         
 3   batter       315 non-null    object        
 4   batter_id    315 non-null    int64         
 5   bat_team     315 non-null    object        
 6   bowl_team    315 non-null    object        
 7   runs         315 non-null    int64         
 8   balls        315 non-null    int64         
 9   isOut        315 non-null    int64         
 10  wicketType   315 non-null    object        
 11  venue        315 non-null    object        
 12  league       315 non-null    object        
 13  boundaries   315 non-null    int64         
 14  career_avg   306 non-null    float64       
 15  career_rpm   306 non-null    float64       
 16  career_s

In [40]:
df[df['batter_id'] == 44936].league.nunique() # NOTE - Career columns can have one row of null values for each batter


9

In [41]:
df.columns

Index(['match_id', 'date', 'inns', 'batter', 'batter_id', 'bat_team',
       'bowl_team', 'runs', 'balls', 'isOut', 'wicketType', 'venue', 'league',
       'boundaries', 'career_avg', 'career_rpm', 'career_sr', 'career_bpm',
       'career_bfpm'],
      dtype='object')

# Innings Stats

In [42]:
# Sort data by batter and match date
df.sort_values(by=['batter', 'date','league'], inplace=True)


# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby(['batter_id', 'inns'])['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby(['batter_id', 'inns'])['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby(['batter_id', 'inns']).cumcount()
df['cumulative_balls'] = df.groupby(['batter_id', 'inns'])['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby(['batter_id', 'inns'])['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['inns_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['inns_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['inns_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['inns_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['inns_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
inns_cols = ['inns_avg', 'inns_bfpm', 'inns_sr', 'inns_rpm', 'inns_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in inns_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'inns','league']).head(1).index, inns_cols] = pd.NA

In [43]:
 df[(df['batter_id'] == 793463) & (df['inns'] == 2)&(df['career_avg'].isna())].head(12)

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,...,career_avg,career_rpm,career_sr,career_bpm,career_bfpm,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm
46478,1063062,2016-11-13,2,Rashid Khan,793463,Cumilla Warriors,Khulna Tigers,10,7,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63378,1141232,2018-05-31,2,Rashid Khan,793463,ICC World XI,West Indies,9,7,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97006,1247012,2021-02-21,2,Rashid Khan,793463,Lahore Qalandars,Peshawar Zalmi,27,15,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100703,1252668,2021-07-24,2,Rashid Khan,793463,Trent Rockets,Southern Brave,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
126450,1343941,2023-01-10,2,Rashid Khan,793463,MI Cape Town,Paarl Royals,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127859,1346947,2023-02-09,2,Rashid Khan,793463,MI Emirates,Dubai Capitals,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
136645,1381367,2023-07-17,2,Rashid Khan,793463,MI New York,Texas Super Kings,13,9,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
df[df['batter_id'] == 44936].info() # NOTE - Innings stats can have two rows of null values for each batter, one for each inning

<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 161 to 102592
Data columns (total 24 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   match_id     315 non-null    int64         
 1   date         315 non-null    datetime64[ns]
 2   inns         315 non-null    int64         
 3   batter       315 non-null    object        
 4   batter_id    315 non-null    int64         
 5   bat_team     315 non-null    object        
 6   bowl_team    315 non-null    object        
 7   runs         315 non-null    int64         
 8   balls        315 non-null    int64         
 9   isOut        315 non-null    int64         
 10  wicketType   315 non-null    object        
 11  venue        315 non-null    object        
 12  league       315 non-null    object        
 13  boundaries   315 non-null    int64         
 14  career_avg   306 non-null    float64       
 15  career_rpm   306 non-null    float64       
 16  career_s

# Venue Stats

In [45]:
# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter_id', 'venue'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ven_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ven_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ven_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ven_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ven_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
venue_cols = ['ven_avg', 'ven_bfpm', 'ven_sr', 'ven_rpm', 'ven_bpm']

# making the first val as Null
df.loc[df.groupby(['batter', 'venue']).head(1).index, venue_cols] = pd.NA

In [46]:
df[(df['batter_id'] == 793463) & (df['venue'] == 'MA Chidambaram Stadium')].head()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,...,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm,ven_avg,ven_rpm,ven_sr,ven_bpm,ven_bfpm
76456,1178416,2019-04-23,1,Rashid Khan,793463,Sunrisers Hyderabad,Chennai Super Kings,0,0,0,...,12.066667,3.290909,169.158879,0.509091,1.945455,NaN,NaN,NaN,NaN,NaN
101397,1254060,2021-04-11,2,Rashid Khan,793463,Sunrisers Hyderabad,Kolkata Knight Riders,0,0,0,...,8.730769,2.873418,134.319527,0.379747,2.139241,0.0,0.000000,0.000000,0.000000,0.0
101462,1254063,2021-04-14,2,Rashid Khan,793463,Sunrisers Hyderabad,Royal Challengers Bangalore,17,9,1,...,8.730769,2.837500,134.319527,0.375000,2.112500,0.0,0.000000,0.000000,0.000000,0.0
101528,1254066,2021-04-17,2,Rashid Khan,793463,Sunrisers Hyderabad,Mumbai Indians,0,1,1,...,9.037037,3.012346,137.078652,0.395062,2.197531,17.0,5.666667,188.888889,0.666667,3.0
101638,1254071,2021-04-21,2,Rashid Khan,793463,Sunrisers Hyderabad,Punjab Kings,0,0,0,...,8.714286,2.975610,136.312849,0.390244,2.182927,8.5,4.250000,170.000000,0.500000,2.5


In [47]:
print("No of unique venues played at by Ravindra Jadeja : ",len(df[(df['batter_id'] == 44936)]['venue'].unique()))
df[(df['batter_id'] == 44936)].info() # NOTE - Venue stats can have as many row of null values for each batter as the number of venues played at

No of unique venues played at by Ravindra Jadeja :  66
<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 161 to 102592
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   match_id     315 non-null    int64         
 1   date         315 non-null    datetime64[ns]
 2   inns         315 non-null    int64         
 3   batter       315 non-null    object        
 4   batter_id    315 non-null    int64         
 5   bat_team     315 non-null    object        
 6   bowl_team    315 non-null    object        
 7   runs         315 non-null    int64         
 8   balls        315 non-null    int64         
 9   isOut        315 non-null    int64         
 10  wicketType   315 non-null    object        
 11  venue        315 non-null    object        
 12  league       315 non-null    object        
 13  boundaries   315 non-null    int64         
 14  career_avg   306 non-null    float64       
 15  ca

# Season Stats

In [48]:
df['season'] = df['date'].dt.year

# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter_id', 'season','league'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ssn_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ssn_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ssn_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ssn_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ssn_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
ssn_cols = ['ssn_avg', 'ssn_bfpm', 'ssn_sr', 'ssn_rpm', 'ssn_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in ssn_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'season','league']).head(1).index, ssn_cols] = pd.NA

In [49]:
df[(df['batter_id'] == 44936) & (df['season'] == 2018)].head()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,...,ven_rpm,ven_sr,ven_bpm,ven_bfpm,season,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm
61955,1136563,2018-04-08,1,AB de Villiers,44936,Royal Challengers Bangalore,Kolkata Knight Riders,44,23,1,...,16.428571,151.315789,2.285714,10.857143,2018,NaN,NaN,NaN,NaN,NaN
62076,1136568,2018-04-13,2,AB de Villiers,44936,Royal Challengers Bangalore,Punjab Kings,57,40,1,...,28.916667,155.605381,3.958333,18.583333,2018,44.000000,44.000000,191.304348,6.000000,23.00
62142,1136571,2018-04-15,2,AB de Villiers,44936,Royal Challengers Bangalore,Rajasthan Royals,20,18,1,...,29.489796,155.042918,4.000000,19.020408,2018,50.500000,50.500000,160.317460,6.000000,31.50
62207,1136574,2018-04-17,2,AB de Villiers,44936,Royal Challengers Bangalore,Mumbai Indians,1,2,1,...,30.375000,180.000000,4.750000,16.875000,2018,40.333333,40.333333,149.382716,4.666667,27.00
62318,1136579,2018-04-21,2,AB de Villiers,44936,Royal Challengers Bangalore,Delhi Capitals,90,39,0,...,29.300000,154.210526,3.960000,19.000000,2018,30.500000,30.500000,146.987952,3.500000,20.75


In [52]:
# Filter for Ravindra Jadeja
jadeja_data = df[(df['batter_id'] == 44936)]

# Group by league and count unique seasons
seasons_by_league = jadeja_data.groupby('league')['season'].nunique().sort_values(ascending=False)

print("Number of seasons played by Ravindra Jadeja in each league:")
for league, seasons in seasons_by_league.items():
    print(f"{league}: {seasons} seasons")

# Total across all leagues (as before)
print("\nTotal unique seasons played across all leagues:", len(jadeja_data['season'].unique()))

Number of seasons played by Ravindra Jadeja in each league:
IPL: 14 seasons
T20IS: 12 seasons
MSL: 2 seasons
CSA: 2 seasons
BBL: 1 seasons
BPL: 1 seasons
CPL: 1 seasons
PSL: 1 seasons
T20BLAST: 1 seasons

Total unique seasons played across all leagues: 16


In [53]:
print("No of unique seasons played by Ravindra Jadeja : ",len(df[(df['batter_id'] == 44936)]['season'].unique()))
df[(df['batter_id'] == 44936)].info() # NOTE - Season stats can have as many row of null values for each batter as the number of seasons played

No of unique seasons played by Ravindra Jadeja :  16
<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 161 to 102592
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   match_id     315 non-null    int64         
 1   date         315 non-null    datetime64[ns]
 2   inns         315 non-null    int64         
 3   batter       315 non-null    object        
 4   batter_id    315 non-null    int64         
 5   bat_team     315 non-null    object        
 6   bowl_team    315 non-null    object        
 7   runs         315 non-null    int64         
 8   balls        315 non-null    int64         
 9   isOut        315 non-null    int64         
 10  wicketType   315 non-null    object        
 11  venue        315 non-null    object        
 12  league       315 non-null    object        
 13  boundaries   315 non-null    int64         
 14  career_avg   306 non-null    float64       
 15  care

# Last5 Match Stats

In [54]:
# Sort batting data by batter and match date
df.sort_values(by=['batter', 'date'], inplace=True)

# ------------------------------------------------------------------------------
# 1) batter Last 5 Match Rolling batter's Average
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_dismissals = df.groupby('batter_id')['isOut'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1)

df['last5_avg'] = rolling_runs / rolling_dismissals

# ------------------------------------------------------------------------------
# 2) batter Last 5 Match Rolling Strike Rate
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_balls = df.groupby('batter_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1
          )
df['last5_sr'] = (rolling_runs)*100 / rolling_balls

# ------------------------------------------------------------------------------
# 3) batter Last 5 Match Rolling runs per match
# ------------------------------------------------------------------------------
df['last5_rpm'] = df.groupby('batter_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 4) batter Last 5 Match Rolling ball faced per Match
# ------------------------------------------------------------------------------
df['last5_bfpm'] = df.groupby('batter_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 5) batter Last 5 Match Rolling boundaries faced per Match
# ------------------------------------------------------------------------------
df['last5_bpm'] = df.groupby('batter_id')['boundaries'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

In [55]:
df[(df['batter_id'] == 44936)].head()

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,runs,balls,isOut,...,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm
161,238195,2006-02-24,1,AB de Villiers,44936,South Africa,Australia,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
248,255954,2006-12-01,1,AB de Villiers,44936,South Africa,India,6,4,1,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
323,287853,2007-09-11,2,AB de Villiers,44936,South Africa,West Indies,16,9,1,...,NaN,NaN,NaN,NaN,NaN,6.0,150.000000,3.000000,2.000000,0.5
544,287864,2007-09-15,2,AB de Villiers,44936,South Africa,Bangladesh,14,15,0,...,16.0,16.0,177.777778,2.0,9.0,11.0,169.230769,7.333333,4.333333,1.0
598,287867,2007-09-16,1,AB de Villiers,44936,South Africa,England,18,14,1,...,30.0,15.0,125.000000,1.5,12.0,18.0,128.571429,9.000000,7.000000,1.0


In [56]:
df[(df['batter_id'] == 44936)].info() # NOTE - Last5 stats can have one row of null values for each batter

<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 161 to 102592
Data columns (total 40 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   match_id     315 non-null    int64         
 1   date         315 non-null    datetime64[ns]
 2   inns         315 non-null    int64         
 3   batter       315 non-null    object        
 4   batter_id    315 non-null    int64         
 5   bat_team     315 non-null    object        
 6   bowl_team    315 non-null    object        
 7   runs         315 non-null    int64         
 8   balls        315 non-null    int64         
 9   isOut        315 non-null    int64         
 10  wicketType   315 non-null    object        
 11  venue        315 non-null    object        
 12  league       315 non-null    object        
 13  boundaries   315 non-null    int64         
 14  career_avg   306 non-null    float64       
 15  career_rpm   306 non-null    float64       
 16  career_s

# Cleaning and Exporting

In [57]:
df.drop(columns = ['runs', 'balls', 'isOut', 'wicketType', 'boundaries'], inplace=True)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161026 entries, 66470 to 7401
Data columns (total 35 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   match_id     161026 non-null  int64         
 1   date         161026 non-null  datetime64[ns]
 2   inns         161026 non-null  int64         
 3   batter       161026 non-null  object        
 4   batter_id    161026 non-null  int64         
 5   bat_team     161026 non-null  object        
 6   bowl_team    161026 non-null  object        
 7   venue        161026 non-null  object        
 8   league       161026 non-null  object        
 9   career_avg   152636 non-null  float64       
 10  career_rpm   152636 non-null  float64       
 11  career_sr    152636 non-null  float64       
 12  career_bpm   152636 non-null  float64       
 13  career_bfpm  152636 non-null  float64       
 14  inns_avg     145653 non-null  float64       
 15  inns_rpm     145653 non-null  float64

In [59]:
df['date'].min(), df['date'].max()

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-04-07 00:00:00'))

In [60]:
df.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bat_features.csv", index=False)